In [1]:
import sys
import os

sys.path.append(os.path.abspath('..'))

In [2]:

import pandas as pd
import numpy as  np
import datetime as dt
import pandas as pd
import numpy as np
import datetime as dt

from Python_arq import engines as engs
from sqlalchemy import text
from calendar import day_name
from sqlalchemy import text
from pathlib import Path


eng = engs.get_engine()
text_qry = text(engs.load_qry('qry_olos.sql'))
df = pd.read_sql(text_qry, eng)


In [3]:
dia_semana = {
    'Monday': 'Segunda',
    'Tuesday': 'Terça',
    'Wednesday': 'Quarta',
    'Thursday': 'Quinta',
    'Friday': 'Sexta',
    'Saturday': 'Sábado',
    'Sunday': 'Domingo'
}

df['day_week'] = df['data'].apply(lambda x: day_name[x.weekday()]).map(dia_semana)
df['data'] = pd.to_datetime(df['data'])
df['semana_mes'] = (
    df['data'].dt.day.sub(1) // 7 + 1 
)
df['day_week'] = df['day_week'] + '_W' + df['semana_mes'].astype(str)

In [11]:
## performance bases | Hoje ## 
df_today = df[df['data'].dt.date == dt.datetime.today().date()]
df_today = df_today.groupby(['area','base_type']).agg(
    Total_tentativas = ('tentativas','sum'),
    total_atendidas = ('atendidas','sum')
).reset_index()
df_today['performance'] = (df_today['total_atendidas'] / df_today['Total_tentativas'] * 100).round(2)
df_today = df_today.sort_values('performance', ascending=False)
df_today

,area,base_type,Total_tentativas,total_atendidas,performance
12,Veterinária,inativa,38,2,5.26
6,Nutrição,evento,256,11,4.30
4,Fisioterapia,inativa,433,17,3.93
3,Fisioterapia,evento,345,12,3.48
10,Psicologia,ativa,1658,49,2.96
1,Fisioterapia,Base Leads,186,5,2.69
0,Enfermagem,inativa,1070,27,2.52
7,Nutrição,inativa,162,3,1.85
9,Psicologia,Material Rico,519,9,1.73
5,Medicina,ativa,355,6,1.69


In [6]:
## top10 bases ## 
df_top10 = df.groupby(['area','base_type']).agg(
    total_tentativas = ('tentativas','sum'),
    total_atendidas = ('atendidas','sum')
).reset_index()
df_top10['performance'] = (df_top10['total_atendidas'] / df_top10['total_tentativas'] * 100).round(2)
df_top10 = df_top10.sort_values('performance', ascending=False).head(10)
df_top10

,area,base_type,total_tentativas,total_atendidas,performance
57,Veterinária,ativa,58,6,10.34
25,Multi,Carrinho,66,5,7.58
55,Veterinária,Base Leads,3479,214,6.15
60,Veterinária,legado,167,9,5.39
30,Multi,legado,64,3,4.69
31,Nutrição,Base Leads,5716,257,4.50
28,Multi,evento,10897,467,4.29
26,Multi,Material Rico,3556,149,4.19
36,Pediatria,Base Leads,3153,131,4.15
40,Pediatria,evento,4805,189,3.93
